# Práctica final · Agente RAG sobre informes 10-K

Notebook completo de entrega: corpus, cuatro herramientas, agente, retrieval híbrido, guardrail numérico como middleware, golden set, evaluadores, métricas y comparación baseline vs final.


In [1]:
# Preparación automática del proyecto en Google Colab.
import os
import subprocess
import sys
import shutil
from pathlib import Path

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    URL_REPO = "https://github.com/rodovilllapa210/https---github.com-PRACTICA-AGENTE-RAG.git"
    CARPETA_REPO = Path("/content/MIAX_2026/Practica_Agente_RAG")

    if not (CARPETA_REPO / ".git").is_dir():
        CARPETA_REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--quiet", "--depth", "1", "--branch", "main",
             URL_REPO, str(CARPETA_REPO)],
            check=True,
        )

    # Si alguno de estos ficheros se subió manualmente al panel de Colab,
    # se copia al repo. El golden oficial NO es dependencia de la entrega.
    OPCIONALES_SUBIDOS = (
        "miax_s2.py", "golden_set.jsonl", "golden_set_oficial.jsonl",
        "baseline_registros.jsonl", "v4_hibrido_propio_registros.jsonl",
        "v4_hibrido_oficial_registros.jsonl",
    )
    for nombre in OPCIONALES_SUBIDOS:
        destino = CARPETA_REPO / nombre
        subido = Path("/content") / nombre
        if not destino.is_file() and subido.is_file():
            shutil.copy2(subido, destino)

    NECESARIOS = (
        "miax_s1.py", "miax_s2.py", "corpus_miax_2026.zip",
        "indice_faiss.zip", "golden_set.jsonl",
    )
    faltan = [n for n in NECESARIOS if not (CARPETA_REPO / n).is_file()]
    if faltan:
        raise FileNotFoundError(
            "Faltan archivos necesarios en el repositorio: " + ", ".join(faltan)
        )

    os.chdir(CARPETA_REPO)
    if str(CARPETA_REPO) not in sys.path:
        sys.path.insert(0, str(CARPETA_REPO))
    print("Proyecto preparado en", CARPETA_REPO)
else:
    print("Ejecución local: se usan los archivos de la carpeta actual.")


Proyecto preparado en /content/MIAX_2026/Practica_Agente_RAG


In [2]:
# Instalación. Una sola celda, versiones fijadas, salida silenciada.
# Tarda alrededor de minuto y medio: mientras corre, leed la celda siguiente.
%pip install -q \
  langchain==1.3.18 langchain-core==1.6.1 langgraph==1.2.11 \
  langchain-google-genai==4.3.7 google-genai==2.10.0 langchain-huggingface==1.2.2 \
  sentence-transformers==6.0.1 faiss-cpu==1.15.0 rank-bm25==0.2.2 google-auth==2.49.0
print("Instalación terminada.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.5/53.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 958.0/958.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 64.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 2.10.0 which is incompatible.
Instalación terminada.


## Por qué van fijadas las versiones

Porque LangChain publica cada pocos días y una API que se mueve debajo de un
notebook lo rompe sin tocar una línea de código.

No es una precaución teórica: **los notebooks de la edición anterior de este
curso ya no ejecutan.** `create_react_agent`, `MemorySaver`,
`with_structured_output` y `RunnableWithMessageHistory` eran la API correcta
hace un año y hoy son otra cosa. Lo que veis aquí está verificado contra la
documentación oficial el 2 de septiembre de 2026.

De ahí salen dos hábitos que valen para cualquier proyecto que dependa de un
proveedor de modelos:

1. **Fijad la versión exacta**, no el rango. `>=1.3` no es una versión.
2. **Anotad la fecha de verificación** al lado del pin, para saber cómo de
   viejo es lo que estáis leyendo.


In [3]:
# Clave de Gemini: secreto de Colab o variable de entorno. Nunca se imprime.
import os

HAY_CLAVE = bool(os.environ.get("GEMINI_API_KEY"))

if not HAY_CLAVE:
    try:
        from google.colab import userdata
        clave = userdata.get("GEMINI_API_KEY")
        if clave:
            os.environ["GEMINI_API_KEY"] = clave
            HAY_CLAVE = True
    except Exception:
        pass

print("GEMINI_API_KEY:", "disponible" if HAY_CLAVE else "no disponible")
if not HAY_CLAVE:
    print("Las celdas de datos y métricas funcionan; responder/evaluar requieren clave.")


GEMINI_API_KEY: disponible


In [4]:
# %% Corpus e indice  --------------------------------
# Los dos ZIP os los pasamos nosotros (Drive compartido, aula virtual o el
# panel de ficheros de Colab): son 5,6 MB entre los dos. Nada de descargar
# de EDGAR en vivo, que con treinta cuadernos a la vez acaba en bloqueo.
#
# Si los teneis en Drive:
#     from google.colab import drive; drive.mount("/content/drive")
# y anadid la carpeta a CANDIDATOS.
import hashlib, pathlib, zipfile

PAQUETES = [
    ("corpus_miax_2026.zip", "4233c37fc9e9d12091af7a146063ad70903a3fe51404a485854f4021c63daee4"),
    ("indice_faiss.zip", "6b5610ad8ac6ea50364445d39bb464d993cbd87048fb07c4fe16657d7ac11655"),
]
URL_RESPALDO = ""          # vacio si no estan alojados
DESTINO = pathlib.Path("corpus")

CANDIDATOS = [
    pathlib.Path("."),
    pathlib.Path("/content"),
    pathlib.Path("/content/drive/MyDrive/MIAX_2026"),
    pathlib.Path("/content/drive/Shareddrives/MIAX_2026"),
]


def _sha256(ruta):
    d = hashlib.sha256()
    with open(ruta, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            d.update(b)
    return d.hexdigest()


def _localizar(nombre):
    for base in CANDIDATOS:
        ruta = base / nombre
        if ruta.is_file():
            return ruta
    if URL_RESPALDO:
        import urllib.request
        destino = pathlib.Path(nombre)
        urllib.request.urlretrieve(f"{URL_RESPALDO}/{nombre}", destino)
        return destino
    return None


try:
    for nombre, esperado in PAQUETES:
        origen = _localizar(nombre)
        assert origen is not None, (
            f"No encuentro {nombre}. Subelo con el panel de ficheros de "
            f"Colab (icono de carpeta a la izquierda), o monta el Drive "
            f"donde este. Buscado en: {[str(c) for c in CANDIDATOS]}"
        )
        obtenido = _sha256(origen)
        assert obtenido == esperado, (
            f"{nombre} no coincide con lo esperado: el fichero esta "
            f"corrupto o es de otra version.\n"
            f"  esperado: {esperado}\n  obtenido: {obtenido}"
        )
        with zipfile.ZipFile(origen) as zf:
            zf.extractall(DESTINO)

    # Los dos manifiestos declaran el hash de chunks.jsonl. El indice se
    # construyo sobre ESE fichero: si no cuadra, el indice y sus metadatos
    # estan desalineados y el retrieval devuelve el texto equivocado sin
    # dar ningun error.
    huella = _sha256(DESTINO / "chunks.jsonl")
    for manifiesto in ("MANIFEST.md", "indice/MANIFEST.md"):
        ruta = DESTINO / manifiesto
        if ruta.exists():
            assert huella in ruta.read_text(encoding="utf-8"), (
                f"chunks.jsonl no cuadra con {manifiesto}: el indice se "
                "construyo sobre otros fragmentos."
            )

    print("Corpus e indice verificados en", DESTINO.resolve())
    for p in sorted(DESTINO.rglob("*")):
        if p.is_file():
            rel = str(p.relative_to(DESTINO))
            print(f"  {rel:28s} {p.stat().st_size / 1e6:7.2f} MB")

except Exception as e:
    print("No se pudo preparar el corpus:", e)
    print("Pide los ficheros al profesor y dejalos junto al notebook.")


Corpus e indice verificados en /content/MIAX_2026/Practica_Agente_RAG/corpus
  LEEME.md                        0.00 MB
  MANIFEST.md                     0.00 MB
  chunks.jsonl                    3.80 MB
  indice/MANIFEST.md              0.00 MB
  indice/chunks_meta.parquet      1.48 MB
  indice/corpus.faiss             2.69 MB
  secciones.jsonl                 3.21 MB
  xbrl_facts.parquet              0.01 MB


In [5]:
from langchain.chat_models import init_chat_model

MODELO = "google_genai:gemini-3.8-flash"

modelo = None
if HAY_CLAVE:
    try:
        # Gemini 3.8 Flash: dejamos el muestreo en la configuración
        # recomendada por el proveedor; no fijamos temperature/top_p/top_k.
        modelo = init_chat_model(MODELO)
        print("Modelo preparado:", MODELO)
        print("Sampling: model_default")
    except Exception as e:
        print(f"No se pudo crear el modelo ({type(e).__name__}: {e}).")

import pathlib
assert pathlib.Path("corpus/chunks.jsonl").is_file(), \
    "El corpus no está preparado."
assert pathlib.Path("corpus/indice/corpus.faiss").is_file(), \
    "Falta el índice FAISS."
print("§1 listo.")


Modelo preparado: google_genai:gemini-3.8-flash
Sampling: model_default
§1 listo.


## Anatomía de un 10-K

El 10-K es el informe anual que toda empresa cotizada en EE. UU. presenta ante
la SEC. Es un documento normalizado: los mismos epígrafes, en el mismo orden,
todos los años y en todas las compañías. Eso es lo que lo hace utilizable como
corpus.

Nos quedamos con cuatro epígrafes, que son donde está lo que se puede
preguntar:

| Item | Qué contiene | Qué se le pregunta |
| --- | --- | --- |
| **1A** · Risk Factors | Los riesgos que la compañía declara | Qué riesgos nuevos aparecen, cómo cambian entre ejercicios |
| **7** · MD&A | La dirección explicando sus propios resultados | Por qué subió o bajó una magnitud |
| **7A** · Market Risk | Exposición a tipos, divisa y precios | Cuantitativo y corto |
| **8** · Financial Statements | Los estados financieros y sus notas | Cifras, y de dónde salen |

Dos cosas que hay que saber del corpus antes de tocarlo:

**`fiscal_year` no es el año de presentación.** Las seis compañías cierran
ejercicio en cuatro meses distintos —NVDA en enero, MSFT en junio, AAPL en
septiembre, y GOOGL, META y AMZN en diciembre—, y está elegido así a
propósito. El 10-K de NVDA FY2025 se presentó en febrero de 2025; el de
Alphabet FY2025, en febrero de **2026**. Quien razone por fecha de
presentación se equivoca.

**NVIDIA no pone sus estados financieros bajo el Item 8.** Los deja bajo el
Item 15 y en el 8 escribe una remisión de dos líneas. El corpus sirve el
contenido correcto bajo la clave `"8"` y deja constancia en el campo
`item_origen`. Si no lo hiciera, `read_section("NVDA", 2025, "8")` devolvería
cuarenta tokens inútiles.


In [6]:
# Cuánto ocupa un 10-K. Los tokens vienen precalculados en el corpus: contar
# en vivo tardaría más que la clase entera.
import json
import pandas as pd

secciones = pd.DataFrame(
    json.loads(l) for l in open("corpus/secciones.jsonl", encoding="utf-8")
)

tabla = secciones.pivot_table(
    index=["ticker", "fiscal_year"], columns="item", values="n_tokens"
).astype(int)
tabla["TOTAL"] = tabla.sum(axis=1)
print(tabla.to_string())

print(f"\nCorpus entero: {secciones.n_tokens.sum():,} tokens en "
      f"{len(secciones)} secciones")
print(f"Informe medio: {tabla['TOTAL'].mean():,.0f} tokens")
print(f"Informe mayor: {tabla['TOTAL'].max():,} · menor: "
      f"{tabla['TOTAL'].min():,}")

mayor = secciones.nlargest(1, "n_tokens").iloc[0]
# Ojo con `mayor.item`: en pandas eso es el método Series.item, no la
# columna. Con una columna que se llama 'item' hay que usar corchetes.
print(f"Sección mayor: {mayor['ticker']} FY{mayor['fiscal_year']} "
      f"Item {mayor['item']} con {mayor['n_tokens']:,} tokens")


item                   1A      7    7A      8  TOTAL
ticker fiscal_year                                  
AAPL   2024         11663   3814   612  15999  32088
       2025         11626   4294   612  16358  32890
AMZN   2024         10318   9597  1614  28097  49626
       2025         10516   9034  1547  29103  50200
GOOGL  2024         14727  11947  1877  30380  58931
       2025         14984  10648  1579  31845  59056
META   2024         33573  12621  1128  28501  75823
       2025         34751  12518  1144  32351  80764
MSFT   2024         12650  10295   409  28455  51809
       2025         11793   9510   409  26500  48212
NVDA   2024         18681   8348   635  26909  54573
       2025         19476   7824   638  27209  55147

Corpus entero: 649,119 tokens en 48 secciones
Informe medio: 54,093 tokens
Informe mayor: 80,764 · menor: 32,088
Sección mayor: META FY2025 Item 1A con 34,751 tokens


## Qué es una herramienta, y qué ve el modelo de ella

Una *tool* es una función de Python que el modelo puede pedir que se ejecute.
El decorador `@tool` la convierte en un esquema —nombre, parámetros con sus
tipos, y descripción— y ese esquema viaja en la petición junto a los mensajes.

Lo que hay que entender es **qué parte de vuestra función ve el modelo**:

| Lo ve | No lo ve |
| --- | --- |
| El nombre de la función | El cuerpo |
| Los nombres y tipos de los parámetros | Los comentarios |
| El *docstring*, entero | Cómo de rápida o cara es |
| Lo que devuelve, cuando la llama | Lo que hace por dentro |

De ahí sale la consecuencia que gobierna el resto de la sesión: **el modelo
decide si os llama leyendo el docstring**. Un docstring vago produce un agente
que elige mal, con el mismo código debajo. Volveremos a esto en el segundo
ejercicio.

Empezamos por la herramienta fácil: exacta, determinista y prácticamente
gratis.


In [7]:
from langchain.tools import tool

xbrl = pd.read_parquet("corpus/xbrl_facts.parquet")
print(f"{len(xbrl)} hechos XBRL · {xbrl.concept.nunique()} conceptos "
      f"distintos · {xbrl.ticker.nunique()} compañías")


@tool
def get_xbrl_fact(ticker: str, fiscal_year: int, concept: str) -> str:
    """Devuelve el valor EXACTO de una magnitud financiera tal y como la
    compañía la reportó en XBRL.

    Es la fuente autorizada para magnitudes contables estandarizadas que
    estén disponibles en XBRL (revenue, net income, assets, etc.). No la uses
    para porcentajes narrativos, guidance u otras cifras que solo aparezcan
    en el texto del 10-K.

    Args:
        ticker: Símbolo bursátil, p. ej. 'NVDA'.
        fiscal_year: Ejercicio fiscal reportado, p. ej. 2024.
        concept: Concepto US-GAAP, p. ej. 'Revenues', 'NetIncomeLoss',
            'Assets', 'OperatingIncomeLoss'.

    Devuelve el valor con su unidad y fecha de cierre, o un aviso explícito
    si la compañía no reportó ese concepto en ese ejercicio.
    """
    filas = xbrl[
        (xbrl.ticker == ticker)
        & (xbrl.fiscal_year == int(fiscal_year))
        & (xbrl.concept == concept)
    ]

    if filas.empty:
        disponibles = sorted(
            xbrl[
                (xbrl.ticker == ticker)
                & (xbrl.fiscal_year == int(fiscal_year))
            ].concept.unique()
        )
        if not disponibles:
            return (
                f"No hay datos de {ticker} para FY{fiscal_year} en el corpus. "
                "Usa list_available para ver qué hay."
            )
        return (
            f"{ticker} no reportó '{concept}' en FY{fiscal_year}. "
            f"Conceptos disponibles: {', '.join(disponibles)}"
        )

    f = filas.iloc[0]
    return (
        f"{ticker} FY{fiscal_year} · {concept} = {f.value:,.0f} {f.unit} "
        f"(cierre de ejercicio {f.period_end}, según el {f.form})"
    )


135 hechos XBRL · 13 conceptos distintos · 6 compañías


In [8]:
import numpy as np
import miax_s2


# Recursos del retriever híbrido. Las funciones de miax_s2 están cacheadas,
# así que el índice, el encoder y BM25 se cargan una sola vez por sesión.
INDICE_HIBRIDO, META_HIBRIDA, _ = miax_s2.cargar_indice()
BM25_HIBRIDO, CHUNKS_BM25 = miax_s2.montar_bm25()
KK_RRF = 60


def _buscar_hibrido_filtrado(
    query: str,
    ticker: str | None = None,
    fiscal_year: int | None = None,
    item: str | None = None,
    k: int = 5,
) -> list[dict]:
    """Dense + BM25 con filtros de metadatos y Reciprocal Rank Fusion.

    La consulta ya la formula el agente. No se añade una segunda llamada a
    un LLM para reescribirla: V3 mostró que el propio agente ya genera
    consultas en inglés útiles y que un rewriting adicional puede introducir
    regresiones.
    """
    # ---------- ranking denso sobre todo el índice ---------------------
    puntuaciones, posiciones = INDICE_HIBRIDO.search(
        miax_s2.codificar([query]),
        INDICE_HIBRIDO.ntotal,
    )

    densos = []
    for puntuacion, posicion in zip(puntuaciones[0], posiciones[0]):
        if int(posicion) < 0:
            continue

        fila = META_HIBRIDA.iloc[int(posicion)]

        if ticker is not None and fila["ticker"] != ticker:
            continue
        if (
            fiscal_year is not None
            and int(fila["fiscal_year"]) != int(fiscal_year)
        ):
            continue
        if item is not None and fila["item"] != item:
            continue

        densos.append(
            miax_s2.fila_a_fragmento(fila, puntuacion)
        )

    if not densos:
        return []

    rango_denso = {
        frag["chunk_id"]: pos
        for pos, frag in enumerate(densos, start=1)
    }
    permitidos = set(rango_denso)

    # ---------- ranking léxico BM25, restringido a los mismos filtros --
    scores_bm25 = BM25_HIBRIDO.get_scores(
        miax_s2.tokenizar(query)
    )
    orden_bm25 = np.argsort(-scores_bm25)

    rango_bm25 = {}
    pos_filtrada = 0
    for idx in orden_bm25:
        chunk_id = CHUNKS_BM25[int(idx)]["chunk_id"]
        if chunk_id not in permitidos:
            continue
        pos_filtrada += 1
        rango_bm25[chunk_id] = pos_filtrada

    # ---------- Reciprocal Rank Fusion --------------------------------
    scores_rrf = {}
    for chunk_id in permitidos:
        rd = rango_denso[chunk_id]
        rb = rango_bm25.get(chunk_id, 10**6)

        scores_rrf[chunk_id] = (
            1.0 / (KK_RRF + rd)
            + 1.0 / (KK_RRF + rb)
        )

    orden = sorted(
        scores_rrf,
        key=scores_rrf.get,
        reverse=True,
    )

    por_id = {f["chunk_id"]: f for f in densos}
    salida = []

    for chunk_id in orden[: int(k)]:
        frag = dict(por_id[chunk_id])
        frag["puntuacion"] = round(
            float(scores_rrf[chunk_id]),
            6,
        )
        salida.append(frag)

    return salida


def _formatear_hibrido(fragmentos: list[dict]) -> str:
    """Formato estable para el agente; conserva chunk_id verificable."""
    if not fragmentos:
        return (
            "Sin resultados para esa consulta con esos filtros. "
            "Prueba a quitar algún filtro o a reformular la búsqueda."
        )

    return "\n\n---\n\n".join(
        f"[{f['chunk_id']}] {f['ticker']} FY{f['fiscal_year']} "
        f"Item {f['item']} (RRF {f['puntuacion']:.6f})\n{f['texto']}"
        for f in fragmentos
    )


@tool
def search_filings(
    query: str,
    ticker: str | None = None,
    fiscal_year: int | None = None,
    item: str | None = None,
    k: int = 5,
) -> str:
    """Busca fragmentos relevantes en los 10-K mediante retrieval híbrido.

    Combina búsqueda semántica densa y BM25 mediante Reciprocal Rank Fusion,
    aplicando filtros de ticker, ejercicio e Item cuando se proporcionan.

    Úsala para riesgos, estrategia, litigios, comentarios de la dirección
    y cifras narrativas que solo aparecen en el texto (guidance, porcentajes
    segmentarios, capex comentado, etc.). Para magnitudes contables
    estandarizadas disponibles en XBRL, usa get_xbrl_fact.

    Args:
        query: Consulta de búsqueda, preferiblemente en inglés.
        ticker: Filtra por compañía si la pregunta la menciona.
        fiscal_year: Filtra por ejercicio si la pregunta lo menciona.
        item: '1A', '7', '7A' u '8'.
        k: Número de fragmentos a devolver.

    Devuelve k fragmentos con chunk_id para poder citarlos.
    """
    return _formatear_hibrido(
        _buscar_hibrido_filtrado(
            query=query,
            ticker=ticker,
            fiscal_year=fiscal_year,
            item=item,
            k=k,
        )
    )


print("search_filings V4: dense + BM25/RRF + filtros de metadatos.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

search_filings V4: dense + BM25/RRF + filtros de metadatos.


In [9]:
@tool
def read_section(ticker: str, fiscal_year: int, item: str) -> str:
    """Devuelve el TEXTO COMPLETO de una sección de un 10-K.

    Es una herramienta CARA. Úsala solo cuando search_filings no proporcione
    contexto suficiente y sea necesario leer una sección completa.

    Args:
        ticker: Símbolo bursátil.
        fiscal_year: Ejercicio fiscal.
        item: '1A', '7', '7A' u '8'.
    """
    filas = secciones[
        (secciones.ticker == ticker)
        & (secciones.fiscal_year == int(fiscal_year))
        & (secciones.item == item)
    ]

    if filas.empty:
        return (
            f"No hay Item {item} de {ticker} FY{fiscal_year} en el corpus. "
            "Usa list_available para ver qué hay."
        )

    return filas.iloc[0].texto


In [10]:
@tool
def list_available() -> str:
    """Lista qué compañías, ejercicios y secciones existen en el corpus.

    Úsala antes de responder que un dato no existe y cuando no estés seguro
    de que la compañía o el ejercicio solicitado estén disponibles.
    """
    orden_items = {"1A": 0, "7": 1, "7A": 2, "8": 3}
    lineas = []

    for ticker, filas in secciones.groupby("ticker", sort=True):
        empresas = ", ".join(
            sorted(filas["empresa"].dropna().astype(str).unique())
        )
        ejercicios = ", ".join(
            f"FY{anio}"
            for anio in sorted(
                filas["fiscal_year"].dropna().astype(int).unique()
            )
        )
        items = sorted(
            filas["item"].dropna().astype(str).unique(),
            key=lambda item: (orden_items.get(item, len(orden_items)), item),
        )
        lineas.append(
            f"{ticker} — {empresas}: {ejercicios} · Items: {', '.join(items)}"
        )

    return "\n".join(lineas)


HERRAMIENTAS = [
    list_available,
    get_xbrl_fact,
    search_filings,
    read_section,
]
NOMBRES_HERRAMIENTAS = {tool.name for tool in HERRAMIENTAS}

print("Herramientas:", ", ".join(t.name for t in HERRAMIENTAS))


Herramientas: list_available, get_xbrl_fact, search_filings, read_section


## Agente final

La versión final mantiene las cuatro firmas contractuales, usa salida estructurada, límites de llamadas y un guardrail numérico implementado como `after_model` middleware.


### V4 — Retrieval híbrido y trazabilidad robusta

Esta variante mantiene el corpus, las cuatro firmas de herramientas, los dos
golden sets, el guardrail numérico, los límites de llamadas y las métricas de
S2.

Cambia únicamente dos elementos del sistema end-to-end:

1. `search_filings` pasa de dense+filtros a **dense + BM25 + RRF**, conservando
   los mismos filtros de metadatos y la misma firma pública.
2. Si `fuente` es `"texto"` o `"ambas"`, el contrato estructurado exige
   `cita` y `chunk_id` no nulos.

No se añade query rewriting explícito al agente. V3 mostró que el propio agente
ya formula consultas en inglés y que una segunda reescritura con LLM añade
coste y puede introducir regresiones.

Gemini sigue usando `model_default`: no se fuerza `temperature`.


In [11]:
SYSTEM = """Eres un analista financiero que responde preguntas sobre informes
10-K usando ÚNICAMENTE las herramientas disponibles.

Reglas de fuentes:
- Para magnitudes contables estandarizadas disponibles en XBRL (revenue,
  net income, gross profit, assets, operating income, R&D, cash flow, etc.),
  usa get_xbrl_fact. No leas esas magnitudes de la prosa.
- Para riesgos, estrategia, comentarios de la dirección, guidance,
  porcentajes narrativos, mix de segmentos y otras cifras que solo aparecen
  en el texto, usa search_filings.
- Una pregunta puede combinar ambas fuentes: XBRL para la magnitud contable y
  texto para la explicación o cifra narrativa.
- Si no sabes si una compañía o un ejercicio están en el corpus, empieza por
  list_available.
- El corpus está en inglés: escribe las consultas de búsqueda en inglés.
- Si el dato no está en el corpus, dilo. No lo estimes.

Reglas de eficiencia:
- Si la pregunta es exclusivamente sobre una magnitud contable XBRL, obtén
  la cifra con get_xbrl_fact y responde. No uses search_filings ni
  read_section para confirmarla.
- Usa search_filings con k=5 por defecto. Aumenta k o repite la búsqueda solo
  si la evidencia recuperada es insuficiente.
- En cuanto tengas evidencia suficiente para responder, deja de buscar.
- read_section es un último recurso cuando search_filings no aporta contexto
  suficiente.

Reglas para preguntas comparativas:
- Separa la parte numérica XBRL de la parte textual.
- Para comparar una magnitud XBRL entre ejercicios, consulta
  get_xbrl_fact para cada ejercicio necesario.
- En el campo estructurado `cifra`, devuelve SIEMPRE el valor de la magnitud
  principal en el ejercicio final solicitado. La variación absoluta y el
  porcentaje de cambio deben aparecer en `respuesta`, no en `cifra`.
- Para afirmar que un riesgo, iniciativa o descripción cambió entre
  ejercicios, consulta el texto de los ejercicios necesarios cuando la
  comparación lo requiera.
- No sigas buscando una vez que la comparación esté sustentada.

Reglas para cifras que solo están en texto:
- Si la pregunta es extractiva y la cifra solicitada solo existe en la prosa
  (por ejemplo guidance o un porcentaje narrativo), inclúyela en `respuesta`
  y en la cita, pero deja `cifra=None` salvo que exista una magnitud XBRL
  principal que la pregunta también solicite.

Reglas de cita:
- Cita el chunk_id del fragmento textual en el que se apoya la afirmación
  cualitativa o narrativa principal.
- En una pregunta que combine XBRL y texto, `cita` y `chunk_id` deben
  corresponder a la evidencia textual, no a una cifra respaldada por XBRL.
- Si `fuente` es "texto" o "ambas", `cita` y `chunk_id` son OBLIGATORIOS:
  no pueden ser nulos ni vacíos y deben corresponder al mismo fragmento
  devuelto por search_filings.
- Si `fuente` es "xbrl" o "ninguna", `cita` y `chunk_id` pueden ser nulos.
"""


In [12]:
# Contrato de respuesta V4: mantiene los mismos campos de la sesión 1 y
# añade una invariante de trazabilidad.
from typing import Literal

from pydantic import BaseModel, Field, model_validator


class RespuestaFinanciera(BaseModel):
    """Respuesta trazable a una pregunta sobre informes 10-K."""

    respuesta: str = Field(
        description="Respuesta en prosa, breve y directa"
    )
    cifra: float | None = Field(
        default=None,
        description=(
            "Magnitud numérica principal cuando procede. En comparativas, "
            "debe ser el valor de esa magnitud en el ejercicio final "
            "solicitado; diferencias y porcentajes van en `respuesta`. "
            "En extractivas con cifras solo narrativas puede ser None."
        ),
    )
    unidad: str | None = Field(
        default=None,
        description="USD, shares, porcentaje…",
    )
    ticker: str | None = None
    ejercicio: int | None = Field(
        default=None,
        description=(
            "Ejercicio al que corresponde `cifra`; en comparativas, "
            "el ejercicio final solicitado."
        ),
    )
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"] = Field(
        description=(
            "De dónde sale el dato. 'ninguna' si no está en el corpus."
        )
    )
    cita: str | None = Field(
        default=None,
        description=(
            "Texto literal del informe. OBLIGATORIO si fuente es "
            "'texto' o 'ambas'."
        ),
    )
    chunk_id: str | None = Field(
        default=None,
        description=(
            "Identificador del fragmento citado. OBLIGATORIO si fuente es "
            "'texto' o 'ambas'."
        ),
    )

    @model_validator(mode="after")
    def validar_trazabilidad_textual(self):
        if self.fuente in {"texto", "ambas"}:
            if not self.cita or not self.cita.strip():
                raise ValueError(
                    "Si fuente es 'texto' o 'ambas', `cita` es obligatoria."
                )
            if not self.chunk_id or not self.chunk_id.strip():
                raise ValueError(
                    "Si fuente es 'texto' o 'ambas', `chunk_id` es obligatorio."
                )
        return self


print(json.dumps(
    RespuestaFinanciera.model_json_schema()["properties"],
    indent=2,
    ensure_ascii=False,
)[:1100], "...")
print(
    "Contrato V4: texto/ambas => cita y chunk_id obligatorios."
)


{
  "respuesta": {
    "description": "Respuesta en prosa, breve y directa",
    "title": "Respuesta",
    "type": "string"
  },
  "cifra": {
    "anyOf": [
      {
        "type": "number"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "Magnitud numérica principal cuando procede. En comparativas, debe ser el valor de esa magnitud en el ejercicio final solicitado; diferencias y porcentajes van en `respuesta`. En extractivas con cifras solo narrativas puede ser None.",
    "title": "Cifra"
  },
  "unidad": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "description": "USD, shares, porcentaje…",
    "title": "Unidad"
  },
  "ticker": {
    "anyOf": [
      {
        "type": "string"
      },
      {
        "type": "null"
      }
    ],
    "default": null,
    "title": "Ticker"
  },
  "ejercicio": {
    "anyOf": [
      {
        "type": "integer"
      },
 

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentState,
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    after_model,
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime

MARCA_GUARDRAIL = "VERIFICACIÓN AUTOMÁTICA"
TOLERANCIA_GUARDRAIL = 0.01


def _contenido_mensaje(mensaje) -> str:
    if isinstance(mensaje, dict):
        return str(mensaje.get("content", ""))
    return str(getattr(mensaje, "content", ""))


@after_model(can_jump_to=["model"])
def verificar_cifras_contra_xbrl(
    state: AgentState,
    runtime: Runtime,
) -> dict | None:
    """Comprueba la cifra estructurada contra XBRL y permite una corrección.

    Solo aplica cuando la respuesta declara fuente XBRL/ambas. Si la cifra no
    coincide con ningún hecho XBRL del ticker y ejercicio declarados, devuelve
    el desajuste al modelo y salta una única vez de nuevo a `model`.
    """
    respuesta = state.get("structured_response")
    if respuesta is None or getattr(respuesta, "cifra", None) is None:
        return None

    if getattr(respuesta, "fuente", None) not in {"xbrl", "ambas"}:
        return None

    ticker = getattr(respuesta, "ticker", None)
    ejercicio = getattr(respuesta, "ejercicio", None)
    if ticker is None or ejercicio is None:
        return None

    # Una única corrección por invocación: evita bucles model -> verifier.
    if any(
        MARCA_GUARDRAIL in _contenido_mensaje(m)
        for m in state.get("messages", [])
    ):
        return None

    hechos = xbrl[
        (xbrl["ticker"] == ticker)
        & (xbrl["fiscal_year"] == int(ejercicio))
    ]
    if hechos.empty:
        return None

    afirmada = float(respuesta.cifra)
    if any(
        miax_s2.cuadra(afirmada, float(real), TOLERANCIA_GUARDRAIL)
        for real in hechos["value"]
    ):
        return None

    reportados = [
        f"{fila.concept}={float(fila.value):,.0f} {fila.unit}"
        for fila in hechos.itertuples()
    ]
    mensaje = (
        f"{MARCA_GUARDRAIL}: afirmaste {afirmada:,.6g} para "
        f"{ticker} FY{int(ejercicio)}, pero no coincide (tolerancia "
        f"{TOLERANCIA_GUARDRAIL:.0%}) con ningún hecho XBRL reportado. "
        "Hechos XBRL disponibles para ese ejercicio: "
        + "; ".join(reportados)
        + ". Corrige la respuesta usando XBRL o indica que la cifra solicitada "
          "no está disponible en el corpus. Mantén cita/chunk_id si respaldan "
          "la parte textual."
    )
    return {
        "messages": [{"role": "user", "content": mensaje}],
        "jump_to": "model",
    }


limite_tools = ToolCallLimitMiddleware(run_limit=11, exit_behavior="end")
limite_modelo = ModelCallLimitMiddleware(run_limit=10)

agente = None
if HAY_CLAVE:
    agente = create_agent(
        model=modelo,
        tools=HERRAMIENTAS,
        system_prompt=SYSTEM,
        response_format=RespuestaFinanciera,
        checkpointer=InMemorySaver(),
        middleware=[
            limite_tools,       # límites primero
            limite_modelo,
            verificar_cifras_contra_xbrl,
        ],
    )
    print("Agente final montado con guardrail after_model.")
else:
    print("Sin clave: el agente no se monta, pero el notebook sigue siendo inspeccionable.")


Agente final montado con guardrail after_model.


In [14]:
# ---------------------------------------------------------------------
# Baseline reproducible: misma tarea, búsqueda densa original.
# ---------------------------------------------------------------------
import miax_s1

def _search_filings_baseline_impl(
    query: str,
    ticker: str | None = None,
    fiscal_year: int | None = None,
    item: str | None = None,
    k: int = 5,
) -> str:
    """Busca fragmentos relevantes con el retriever denso del baseline.

    Args:
        query: Consulta en lenguaje natural, preferiblemente en inglés.
        ticker: Filtro opcional por ticker.
        fiscal_year: Filtro opcional por ejercicio fiscal.
        item: Filtro opcional por sección 10-K.
        k: Número de fragmentos a devolver.

    Returns:
        Fragmentos formateados con su chunk_id y metadatos.
    """
    return miax_s1.formatear_fragmentos(
        miax_s1.buscar(
            query,
            ticker=ticker,
            fiscal_year=fiscal_year,
            item=item,
            k=k,
        )
    )

# La firma pública sigue llamándose search_filings para que el evaluador de
# trayectoria vea exactamente la misma tool.
search_filings_baseline = tool("search_filings")(_search_filings_baseline_impl)

HERRAMIENTAS_BASELINE = [
    list_available,
    get_xbrl_fact,
    search_filings_baseline,
    read_section,
]

SYSTEM_BASELINE = """Eres un analista financiero que responde preguntas sobre
informes 10-K usando ÚNICAMENTE las herramientas disponibles.

Reglas:
- Para cualquier CIFRA, usa get_xbrl_fact. Nunca leas un número de la prosa.
- Para riesgos, estrategia o comentarios de la dirección, usa search_filings.
- Si no sabes si una compañía o un ejercicio están en el corpus, empieza por
  list_available.
- El corpus está en inglés: escribe las consultas de búsqueda en inglés.
- Cita el chunk_id del fragmento en el que te apoyes.
- Si el dato no está en el corpus, dilo. No lo estimes.
"""


class RespuestaFinancieraBaseline(BaseModel):
    """Contrato original usado por el baseline."""

    respuesta: str
    cifra: float | None = None
    unidad: str | None = None
    ticker: str | None = None
    ejercicio: int | None = None
    fuente: Literal["xbrl", "texto", "ambas", "ninguna"]
    cita: str | None = None
    chunk_id: str | None = None


agente_baseline = None
if HAY_CLAVE:
    limite_tools_baseline = ToolCallLimitMiddleware(
        run_limit=11,
        exit_behavior="end",
    )
    agente_baseline = create_agent(
        model=modelo,
        tools=HERRAMIENTAS_BASELINE,
        system_prompt=SYSTEM_BASELINE,
        response_format=RespuestaFinancieraBaseline,
        checkpointer=InMemorySaver(),
        middleware=[
            limite_tools_baseline,
            verificar_cifras_contra_xbrl,
        ],
    )
    print("Baseline reproducible preparado (dense + filtros, sin BM25/RRF).")
else:
    print("Sin clave: el baseline no se monta.")


Baseline reproducible preparado (dense + filtros, sin BM25/RRF).


## La práctica

El enunciado completo lo tenéis en la mano. Aquí queda lo imprescindible.

**Qué se entrega**, en grupos de 3:

| | Peso |
| --- | --- |
| Repositorio con el agente, el golden set y los evaluadores | 30 |
| Presentación de 8 minutos el 24, con 10 preguntas ciegas en directo | 70 |

**El golden set.** Recibís 20 preguntas oficiales y escribís **20 vuestras**,
de las cuales **al menos 6 comparativas**. Tres familias:

| Familia | Qué mide | Campos que se rellenan |
| --- | --- | --- |
| `extractiva` | Retrieval y trazabilidad | `item_esperado`, `ancla_texto` |
| `numerica` | El guardrail contra XBRL | `cifra_esperada`, `unidad`, `concept_xbrl` |
| `comparativa` | Que el agente descomponga y compare | los tres, por cada ejercicio |

**Por qué el mínimo de 6 comparativas.** «¿Qué riesgos nuevos añadió entre
FY2024 y FY2025?» no la contesta una sola recuperación: hay que descomponer,
recuperar dos veces y comparar. Es la familia donde el agente deja de ser
decoración sobre una *pipeline*, y sin ella vuestro informe no puede demostrar
que hiciera falta un agente. Hay sustancia real que encontrar: el Item 1A
cambia entre el 49 % y el 85 % de los párrafos entre los dos ejercicios, según
la compañía.

**La verdad se ancla a una frase, no a un `chunk_id`.** El día 17 vais a
cambiar el troceado, y en cuanto lo toquéis todos los `chunk_id` son otros. Si
la métrica dependiera de ellos, el grupo que mejorase el troceado saldría
penalizado por haberlo mejorado. Por eso `ancla_texto` es un texto literal del
informe: **una frase**, no tres párrafos.


In [15]:
# Golden propio obligatorio; golden oficial opcional para diagnóstico.
from pathlib import Path

RUTA_GOLDEN_PROPIO = Path("golden_set.jsonl")
RUTA_GOLDEN_OFICIAL = Path("golden_set_oficial.jsonl")

if not RUTA_GOLDEN_PROPIO.is_file():
    raise FileNotFoundError("Falta golden_set.jsonl junto al notebook / en el repo.")


def leer_jsonl(ruta):
    return [
        json.loads(linea)
        for linea in Path(ruta).read_text(encoding="utf-8").splitlines()
        if linea.strip()
    ]


golden = leer_jsonl(RUTA_GOLDEN_PROPIO)
golden_oficial = leer_jsonl(RUTA_GOLDEN_OFICIAL) if RUTA_GOLDEN_OFICIAL.is_file() else []

print(f"Golden propio: {len(golden)} preguntas")
print(f"Golden oficial opcional: {len(golden_oficial)} preguntas")


Golden propio: 20 preguntas
Golden oficial opcional: 20 preguntas


In [16]:
# Vuestras 20 preguntas, y el validador que tienen que pasar antes de
# entregarlas. Un golden set que no pasa el validador no se corrige: se
# devuelve.
PLANTILLA = {
    "id": "g3-001",
    "pregunta": "¿Cuál fue el revenue de NVIDIA en el ejercicio 2024?",
    "familia": "numerica",              # extractiva | numerica | comparativa
    "ticker": "NVDA",
    "fiscal_year": 2024,
    "respuesta_esperada": "60.922 millones de dólares",
    "cifra_esperada": 60922000000.0,
    "unidad": "USD",
    "concept_xbrl": "Revenues",
    "item_esperado": None,
    "ancla_texto": None,
    "ancla_inicio": None,
    "ancla_fin": None,
    "chunk_id_esperado": None,
    "herramienta_esperada": ["get_xbrl_fact"],
    "autor": "grupo-3",
}

CAMPOS = set(PLANTILLA)
FAMILIAS = {"extractiva", "numerica", "comparativa"}


def validar(preguntas: list[dict], exigir_20: bool = True) -> list[str]:
    """Los problemas del fichero, uno por línea. Lista vacía = correcto."""
    problemas = []
    tickers = set(secciones.ticker)
    ejercicios = set(secciones.fiscal_year.astype(int))
    vistos = set()

    for p in preguntas:
        pid = p.get("id", "(sin id)")
        if faltan := CAMPOS - set(p):
            problemas.append(f"{pid}: faltan campos {sorted(faltan)}")
            continue
        if p["id"] in vistos:
            problemas.append(f"{pid}: id repetido")
        vistos.add(p["id"])
        if p["familia"] not in FAMILIAS:
            problemas.append(f"{pid}: familia '{p['familia']}' no válida")
        if p["ticker"] not in tickers:
            problemas.append(f"{pid}: {p['ticker']} no está en el corpus")
        if int(p["fiscal_year"]) not in ejercicios:
            problemas.append(f"{pid}: FY{p['fiscal_year']} no está en el "
                             f"corpus")
        if p["familia"] in {"numerica", "comparativa"}:
            if p.get("cifra_esperada") is None:
                problemas.append(f"{pid}: numérica sin cifra_esperada")
            concepto = p.get("concept_xbrl")
            hay = xbrl[(xbrl.ticker == p["ticker"])
                       & (xbrl.fiscal_year == int(p["fiscal_year"]))
                       & (xbrl.concept == concepto)]
            if concepto and hay.empty:
                problemas.append(
                    f"{pid}: {p['ticker']} no reporta '{concepto}' en "
                    f"FY{p['fiscal_year']}. El concepto se mira en "
                    f"xbrl_facts.parquet, nunca por analogía con otra "
                    f"compañía.")
        if p["familia"] in {"extractiva", "comparativa"}:
            ancla = p.get("ancla_texto")
            if not ancla:
                problemas.append(f"{pid}: extractiva sin ancla_texto")
            elif len(ancla.split()) > 40:
                problemas.append(
                    f"{pid}: ancla de {len(ancla.split())} palabras. Una "
                    f"frase. Así no medís vuestro retrieval, medís vuestro "
                    f"tamaño de ventana.")
        if not p.get("herramienta_esperada"):
            problemas.append(f"{pid}: sin herramienta_esperada")

    if exigir_20:
        if len(preguntas) != 20:
            problemas.append(f"hacen falta 20 preguntas, hay {len(preguntas)}")
        n_comp = sum(p.get("familia") == "comparativa" for p in preguntas)
        if n_comp < 6:
            problemas.append(f"hacen falta 6 comparativas, hay {n_comp}")
    return problemas



problemas_propio = validar(golden, exigir_20=True)
problemas_oficial = validar(golden_oficial, exigir_20=True) if golden_oficial else []

print("Validando golden propio:")
print("\n".join(f"  - {p}" for p in problemas_propio) or "  sin problemas")
if golden_oficial:
    print("\nValidando golden oficial (opcional):")
    print("\n".join(f"  - {p}" for p in problemas_oficial) or "  sin problemas")

# --- verificación -------------------------------------------------------
assert not validar([PLANTILLA], exigir_20=False), \
    "La plantilla debería pasar su propio validador."
assert validar([{**PLANTILLA, "ticker": "TSLA"}], exigir_20=False), \
    "El validador tiene que rechazar una compañía que no está en el corpus."
assert not problemas_propio, "golden_set.jsonl no pasa el validador."
if golden_oficial:
    assert not problemas_oficial, "golden_set_oficial.jsonl no pasa el validador."
print("\nGolden propio: OK.")


Validando golden propio:
  sin problemas

Validando golden oficial (opcional):
  sin problemas

Golden propio: OK.


## V3 — Experimentos de retrieval de la sesión 2

Este bloque **no cambia todavía `search_filings` del agente**. Su objetivo es
medir el retriever de forma aislada sobre `golden_set_oficial.jsonl`.

Se comparan exactamente las cinco configuraciones de S2:

1. denso plano;
2. denso + filtros de metadatos;
3. denso + BM25 mediante Reciprocal Rank Fusion (RRF);
4. reescritura de consulta + denso filtrado;
5. reescritura + híbrido.

La verdad de retrieval es `ancla_texto`, usando `miax_s2.acierta()`. Por tanto,
la métrica no depende del `chunk_id` concreto.

### Control experimental añadido

Como dejamos de forzar `temperature=0`, una misma pregunta podría recibir dos
reescrituras distintas. Para no confundir el efecto de la reescritura con el
efecto de BM25, se genera **una sola reescritura por pregunta** y se reutiliza
en las configuraciones 4 y 5.

Además se registran posición del ancla, latencia y coste de la reescritura.


In [17]:
# Retrieval S2: definiciones. Esta celda no llama todavía al LLM.
from collections import defaultdict
import time
import json
import numpy as np
import pandas as pd
import miax_s2

K_RETRIEVAL = 5
KK_RRF = 60

con_ancla_oficial = [
    g for g in golden_oficial if g.get("ancla_texto")
]

indice_s2, meta_s2, _ = miax_s2.cargar_indice()
bm25_s2, chunks_bm_s2 = miax_s2.montar_bm25()
POR_ID_BM25 = {c["chunk_id"]: c for c in chunks_bm_s2}

print(
    f"{len(con_ancla_oficial)} preguntas con ancla · "
    f"{indice_s2.ntotal} vectores"
)


def denso_plano(consulta: str, k: int = K_RETRIEVAL) -> list[dict]:
    """Top-k denso sobre todo el corpus, sin filtros."""
    puntuaciones, posiciones = indice_s2.search(
        miax_s2.codificar([consulta]),
        int(k),
    )
    salida = []
    for puntuacion, posicion in zip(puntuaciones[0], posiciones[0]):
        if int(posicion) < 0:
            continue
        salida.append(
            miax_s2.fila_a_fragmento(
                meta_s2.iloc[int(posicion)],
                puntuacion,
            )
        )
    return salida


def con_filtros(
    consulta: str,
    ticker=None,
    fiscal_year=None,
    item=None,
    k: int = K_RETRIEVAL,
) -> list[dict]:
    """Búsqueda densa descartando candidatos que no cumplen metadatos."""
    puntuaciones, posiciones = indice_s2.search(
        miax_s2.codificar([consulta]),
        indice_s2.ntotal,
    )

    salida = []
    for puntuacion, posicion in zip(puntuaciones[0], posiciones[0]):
        if int(posicion) < 0:
            continue

        fila = meta_s2.iloc[int(posicion)]

        if ticker is not None and fila["ticker"] != ticker:
            continue
        if (
            fiscal_year is not None
            and int(fila["fiscal_year"]) != int(fiscal_year)
        ):
            continue
        if item is not None and fila["item"] != item:
            continue

        salida.append(miax_s2.fila_a_fragmento(fila, puntuacion))
        if len(salida) >= int(k):
            break

    return salida


def hibrido(
    consulta: str,
    ticker=None,
    fiscal_year=None,
    item=None,
    k: int = K_RETRIEVAL,
    kk: int = KK_RRF,
) -> list[dict]:
    """Fusión denso + BM25 por Reciprocal Rank Fusion."""
    # 1) Ranking denso ya filtrado. Pedimos todos los candidatos válidos.
    densos = con_filtros(
        consulta,
        ticker=ticker,
        fiscal_year=fiscal_year,
        item=item,
        k=indice_s2.ntotal,
    )
    if not densos:
        return []

    rango_denso = {
        frag["chunk_id"]: pos
        for pos, frag in enumerate(densos, start=1)
    }
    permitidos = set(rango_denso)

    # 2) Ranking BM25 sobre el corpus; después restringimos a los mismos
    # candidatos que sobreviven a los filtros de metadatos.
    scores_bm25 = bm25_s2.get_scores(miax_s2.tokenizar(consulta))
    orden_bm25 = np.argsort(-scores_bm25)

    rango_bm25 = {}
    pos_filtrada = 0
    for idx in orden_bm25:
        chunk_id = chunks_bm_s2[int(idx)]["chunk_id"]
        if chunk_id not in permitidos:
            continue
        pos_filtrada += 1
        rango_bm25[chunk_id] = pos_filtrada

    # 3) RRF. Los dos rankings contienen los mismos candidatos tras filtrar.
    puntuacion_rrf = {}
    for chunk_id in permitidos:
        rd = rango_denso.get(chunk_id, 10**6)
        rb = rango_bm25.get(chunk_id, 10**6)
        puntuacion_rrf[chunk_id] = (
            1.0 / (kk + rd)
            + 1.0 / (kk + rb)
        )

    orden = sorted(
        puntuacion_rrf,
        key=puntuacion_rrf.get,
        reverse=True,
    )

    por_id_denso = {f["chunk_id"]: f for f in densos}
    salida = []
    for chunk_id in orden[: int(k)]:
        frag = dict(por_id_denso[chunk_id])
        frag["puntuacion"] = round(float(puntuacion_rrf[chunk_id]), 6)
        salida.append(frag)

    return salida


def posicion_ancla_para(golden_item, buscar, consulta: str):
    """Posición del ancla en el ranking completo de una configuración."""
    ordenados = buscar(
        consulta,
        golden_item["ticker"],
        golden_item["fiscal_year"],
        golden_item["item_esperado"],
        indice_s2.ntotal,
    )
    return miax_s2.posicion_del_ancla(golden_item, ordenados)


def medir_configuracion(
    nombre: str,
    buscador,
    consultas_por_id: dict[str, str],
    con_filtros_meta: bool,
):
    """Recall@5, posición del ancla y latencia de retrieval."""
    recuperados_por_id = {}
    filas = []
    t0_total = time.perf_counter()

    for g in con_ancla_oficial:
        consulta = consultas_por_id[g["id"]]
        t0 = time.perf_counter()

        if con_filtros_meta:
            recuperados = buscador(
                consulta,
                g["ticker"],
                g["fiscal_year"],
                g["item_esperado"],
                K_RETRIEVAL,
            )
        else:
            recuperados = buscador(consulta, K_RETRIEVAL)

        latencia = time.perf_counter() - t0
        recuperados_por_id[g["id"]] = recuperados

        hit = miax_s2.acierta(g, recuperados)

        # Posición completa: diagnóstica, no forma parte del recall@5.
        if con_filtros_meta:
            ranking_completo = buscador(
                consulta,
                g["ticker"],
                g["fiscal_year"],
                g["item_esperado"],
                indice_s2.ntotal,
            )
        else:
            ranking_completo = buscador(
                consulta,
                indice_s2.ntotal,
            )

        posicion = miax_s2.posicion_del_ancla(g, ranking_completo)

        filas.append({
            "configuracion": nombre,
            "id": g["id"],
            "ticker": g["ticker"],
            "fiscal_year": g["fiscal_year"],
            "item": g["item_esperado"],
            "consulta": consulta,
            "hit_at_5": bool(hit),
            "posicion_ancla": posicion,
            "latencia_retrieval_s": latencia,
        })

    recall = miax_s2.recall_en_k(
        con_ancla_oficial,
        recuperados_por_id,
    )

    return {
        "configuracion": nombre,
        "recall@5": float(recall),
        "n": len(con_ancla_oficial),
        "fallan": [
            g["id"] for g in con_ancla_oficial
            if not miax_s2.acierta(
                g,
                recuperados_por_id[g["id"]],
            )
        ],
        "latencia_retrieval_total_s": time.perf_counter() - t0_total,
        "latencia_retrieval_media_s": float(
            np.mean([f["latencia_retrieval_s"] for f in filas])
        ),
        "filas": filas,
    }


print("Funciones de retrieval preparadas. Aún no se ha llamado a Gemini.")


13 preguntas con ancla · 1749 vectores
Funciones de retrieval preparadas. Aún no se ha llamado a Gemini.


In [18]:
# Reescritura controlada: una sola reescritura por pregunta, reutilizada.
INSTRUCCION_REESCRITURA = """Reescribe esta pregunta como una consulta de
búsqueda para un índice de informes 10-K en INGLÉS. Usa el vocabulario del
propio informe. Devuelve SOLO la consulta, sin comillas ni explicación."""

PRECIO_REWRITE_INPUT_USD_M = 0.75
PRECIO_REWRITE_OUTPUT_USD_M = 3.75

USAR_REESCRITURA_EN_VIVO = True

RUTA_REESCRITURAS = Path(
    "resultados/v3_retrieval_reescrituras.jsonl"
)
RUTA_REESCRITURAS.parent.mkdir(exist_ok=True)


def _uso_mensaje(mensaje):
    uso = getattr(mensaje, "usage_metadata", None) or {}
    entrada = int(uso.get("input_tokens", 0) or 0)
    salida = int(uso.get("output_tokens", 0) or 0)
    return entrada, salida


def generar_reescrituras(forzar: bool = False):
    """Genera/carga una reescritura fija por pregunta oficial con ancla."""
    if RUTA_REESCRITURAS.is_file() and not forzar:
        filas = [
            json.loads(linea)
            for linea in RUTA_REESCRITURAS.read_text(
                encoding="utf-8"
            ).splitlines()
            if linea.strip()
        ]
        print(
            f"Reescrituras cargadas de disco: {len(filas)}. "
            "No se llamó a Gemini."
        )
        return filas

    filas = []

    for g in con_ancla_oficial:
        pregunta = g["pregunta"]
        inicio = time.perf_counter()

        consulta = None
        fuente = None
        entrada = salida = 0
        error = None

        if USAR_REESCRITURA_EN_VIVO and modelo is not None:
            try:
                mensaje = modelo.invoke([
                    {
                        "role": "system",
                        "content": INSTRUCCION_REESCRITURA,
                    },
                    {
                        "role": "user",
                        "content": pregunta,
                    },
                ])
                consulta = mensaje.text.strip()
                entrada, salida = _uso_mensaje(mensaje)
                fuente = "gemini_model_default"
            except Exception as exc:
                error = f"{type(exc).__name__}: {exc}"

        if not consulta:
            consulta = miax_s2.REESCRITURAS_RESPALDO.get(
                g["id"],
                pregunta,
            )
            fuente = "respaldo_miax_s2"

        latencia = time.perf_counter() - inicio
        coste = (
            entrada * PRECIO_REWRITE_INPUT_USD_M
            + salida * PRECIO_REWRITE_OUTPUT_USD_M
        ) / 1_000_000

        filas.append({
            "id": g["id"],
            "pregunta": pregunta,
            "consulta": consulta,
            "fuente": fuente,
            "input_tokens": entrada,
            "output_tokens": salida,
            "coste_usd": coste,
            "latencia_s": latencia,
            "error": error,
        })

    with RUTA_REESCRITURAS.open("w", encoding="utf-8") as f:
        for fila in filas:
            f.write(json.dumps(fila, ensure_ascii=False) + "\n")

    return filas


print(
    "Preparado. La siguiente celda generará como máximo "
    f"{len(con_ancla_oficial)} reescrituras, una por pregunta."
)


Preparado. La siguiente celda generará como máximo 13 reescrituras, una por pregunta.


### Experimentos V3 conservados como referencia

La celda siguiente conserva el experimento aislado dense → metadatos → BM25 →
rewriting → rewriting+BM25, pero **ya no lo ejecuta automáticamente**.

Para reproducirlo de nuevo:

```python
ejecutar_experimentos_retrieval_v3()
```

Así un `Run all` de V4 no gasta otras 13 llamadas de rewriting.


In [19]:
def ejecutar_experimentos_retrieval_v3():
    """Reproduce bajo demanda los cinco experimentos de retrieval V3."""
    # Ejecutar deliberadamente este bloque para medir §1 de S2.
    reescrituras = generar_reescrituras(forzar=False)
    REWRITE_POR_ID = {r["id"]: r["consulta"] for r in reescrituras}

    CONSULTA_ORIGINAL = {
        g["id"]: g["pregunta"]
        for g in con_ancla_oficial
    }

    resultados_retrieval = []

    resultados_retrieval.append(
        medir_configuracion(
            "1 · denso plano",
            denso_plano,
            CONSULTA_ORIGINAL,
            con_filtros_meta=False,
        )
    )

    resultados_retrieval.append(
        medir_configuracion(
            "2 · + filtro de metadatos",
            con_filtros,
            CONSULTA_ORIGINAL,
            con_filtros_meta=True,
        )
    )

    resultados_retrieval.append(
        medir_configuracion(
            "3 · + híbrido BM25",
            hibrido,
            CONSULTA_ORIGINAL,
            con_filtros_meta=True,
        )
    )

    resultados_retrieval.append(
        medir_configuracion(
            "4 · + reescritura de consulta",
            con_filtros,
            REWRITE_POR_ID,
            con_filtros_meta=True,
        )
    )

    resultados_retrieval.append(
        medir_configuracion(
            "5 · reescritura + híbrido",
            hibrido,
            REWRITE_POR_ID,
            con_filtros_meta=True,
        )
    )

    # Coste/latencia de rewriting: mismo overhead para las configuraciones 4 y 5.
    coste_rewrite_total = sum(r["coste_usd"] for r in reescrituras)
    latencia_rewrite_total = sum(r["latencia_s"] for r in reescrituras)
    n_rewrite = len(reescrituras)

    filas_resumen = []
    filas_detalle = []

    for r in resultados_retrieval:
        usa_rewrite = r["configuracion"].startswith(("4 ·", "5 ·"))

        filas_resumen.append({
            "configuracion": r["configuracion"],
            "recall@5": r["recall@5"],
            "fallan": ", ".join(r["fallan"]) if r["fallan"] else "",
            "latencia_retrieval_media_s": r["latencia_retrieval_media_s"],
            "llamadas_llm_por_pregunta": 1 if usa_rewrite else 0,
            "coste_rewrite_medio_usd": (
                coste_rewrite_total / n_rewrite
                if usa_rewrite and n_rewrite
                else 0.0
            ),
            "latencia_total_estimada_media_s": (
                r["latencia_retrieval_media_s"]
                + (
                    latencia_rewrite_total / n_rewrite
                    if usa_rewrite and n_rewrite
                    else 0.0
                )
            ),
        })

        filas_detalle.extend(r["filas"])

    tabla_retrieval = pd.DataFrame(filas_resumen)
    detalle_retrieval = pd.DataFrame(filas_detalle)

    RUTA_RESUMEN_RETRIEVAL = Path(
        "resultados/v3_retrieval_metricas.csv"
    )
    RUTA_DETALLE_RETRIEVAL = Path(
        "resultados/v3_retrieval_detalle.csv"
    )

    tabla_retrieval.to_csv(RUTA_RESUMEN_RETRIEVAL, index=False)
    detalle_retrieval.to_csv(RUTA_DETALLE_RETRIEVAL, index=False)

    display(
        tabla_retrieval.style.format({
            "recall@5": "{:.1%}",
            "latencia_retrieval_media_s": "{:.4f}",
            "coste_rewrite_medio_usd": "${:.6f}",
            "latencia_total_estimada_media_s": "{:.4f}",
        })
    )

    print("\nPosición del ancla por pregunta/configuración:")
    display(
        detalle_retrieval.pivot(
            index="id",
            columns="configuracion",
            values="posicion_ancla",
        )
    )

    mejor_recall = tabla_retrieval["recall@5"].max()
    mejores = tabla_retrieval.loc[
        tabla_retrieval["recall@5"] == mejor_recall,
        "configuracion",
    ].tolist()

    print(
        "\nMejor recall@5:",
        f"{mejor_recall:.1%}",
        "·",
        ", ".join(mejores),
    )
    print("Guardado:", RUTA_RESUMEN_RETRIEVAL)
    print("Guardado:", RUTA_DETALLE_RETRIEVAL)

    assert (
        tabla_retrieval.loc[
            tabla_retrieval["configuracion"] == "2 · + filtro de metadatos",
            "recall@5",
        ].iloc[0]
        >=
        tabla_retrieval.loc[
            tabla_retrieval["configuracion"] == "1 · denso plano",
            "recall@5",
        ].iloc[0]
    ), (
        "El filtro de metadatos ha empeorado el recall. "
        "Revisar ticker/fiscal_year/item."
    )


print("Experimentos V3 conservados. Para repetirlos: ""ejecutar_experimentos_retrieval_v3()")


Experimentos V3 conservados. Para repetirlos: ejecutar_experimentos_retrieval_v3()


### Decisión después de V3

No se incorpora automáticamente ninguna técnica al agente.

Primero se comparan `recall@5`, fallos, posición del ancla, coste y latencia.
Solo la configuración que aporte una mejora medible pasará a convertirse en
el nuevo `search_filings`.

Esto mantiene aislado el efecto de cada intervención y evita atribuir a BM25,
rewriting o al agente una mejora que provenga de otra variable.


## Instrumentación, guardrail y métricas

El guardrail ya vive dentro del grafo como `after_model`. La instrumentación solo registra si actuó y verifica el estado final; no realiza una segunda corrección externa.

Se conservan las métricas comparables de S2: cita, cifra, trayectoria, recall@5, coste, latencia y llamadas.


In [20]:
import math
import time
import uuid

try:
    import miax_s2
except ImportError as exc:
    raise ImportError(
        "Falta miax_s2.py. Déjalo junto al notebook o en el repositorio."
    ) from exc

PRECIO_INPUT_USD_M = 0.75
PRECIO_CACHE_READ_USD_M = 0.075
PRECIO_OUTPUT_USD_M = 3.75
TOLERANCIA_OFICIAL = 0.01


# ---------- trazabilidad y uso ------------------------------------------

def llamadas_operativas(resultado):
    """Solo las cuatro herramientas de negocio del contrato."""
    llamadas = []
    for mensaje in resultado.get("messages", []):
        for call in getattr(mensaje, "tool_calls", []) or []:
            if call.get("name") in NOMBRES_HERRAMIENTAS:
                llamadas.append(call.get("name"))
    return llamadas


def uso_tokens(resultado):
    """Uso reportado por todas las llamadas al modelo."""
    totales = {
        "input_tokens": 0,
        "cache_read_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
        "reasoning_tokens": 0,
        "model_calls": 0,
    }

    for mensaje in resultado.get("messages", []):
        usage = getattr(mensaje, "usage_metadata", None)
        if not usage:
            continue

        totales["model_calls"] += 1
        totales["input_tokens"] += usage.get("input_tokens", 0) or 0
        totales["output_tokens"] += usage.get("output_tokens", 0) or 0
        totales["total_tokens"] += usage.get("total_tokens", 0) or 0

        input_details = usage.get("input_token_details") or {}
        totales["cache_read_tokens"] += input_details.get("cache_read", 0) or 0

        output_details = usage.get("output_token_details") or {}
        totales["reasoning_tokens"] += output_details.get("reasoning", 0) or 0

    return totales


def coste_oficial_usd(resultado):
    """Misma fórmula de S2, adaptada a acceso directo a Gemini."""
    entrada, salida = miax_s2.tokens_de(resultado)
    return (
        entrada * PRECIO_INPUT_USD_M
        + salida * PRECIO_OUTPUT_USD_M
    ) / 1e6


def coste_real_estimado_usd(resultado):
    """Estimación adicional aplicando el descuento observado por cache read."""
    uso = uso_tokens(resultado)
    cache = max(0, int(uso["cache_read_tokens"]))
    entrada_no_cache = max(0, int(uso["input_tokens"]) - cache)

    return (
        entrada_no_cache / 1_000_000 * PRECIO_INPUT_USD_M
        + cache / 1_000_000 * PRECIO_CACHE_READ_USD_M
        + uso["output_tokens"] / 1_000_000 * PRECIO_OUTPUT_USD_M
    )


def trayectoria_operativa(resultado):
    """Llamadas operativas con argumentos y ToolMessage asociado."""
    salidas = {}

    for mensaje in resultado.get("messages", []):
        tool_call_id = getattr(mensaje, "tool_call_id", None)
        if tool_call_id:
            contenido = getattr(mensaje, "content", None)
            salidas[tool_call_id] = (
                contenido if isinstance(contenido, str)
                else str(contenido)
            )

    trayectoria = []
    for mensaje in resultado.get("messages", []):
        for call in getattr(mensaje, "tool_calls", []) or []:
            if call.get("name") not in NOMBRES_HERRAMIENTAS:
                continue
            call_id = call.get("id")
            trayectoria.append({
                "id": call_id,
                "name": call.get("name"),
                "args": call.get("args", {}),
                "output": salidas.get(call_id),
            })

    return trayectoria


# ---------- guardrail numérico (observabilidad) -------------------------

def _respuesta_dict(resultado):
    respuesta = resultado.get("structured_response")
    if hasattr(respuesta, "model_dump"):
        return respuesta.model_dump()
    if isinstance(respuesta, dict):
        return respuesta
    return None


def validar_guardrail_numerico(resultado):
    """Revalida la cifra final con la misma regla determinista del middleware."""
    respuesta = _respuesta_dict(resultado) or {}
    cifra = respuesta.get("cifra")
    ticker = respuesta.get("ticker")
    ejercicio = respuesta.get("ejercicio")
    fuente = respuesta.get("fuente")

    if (
        cifra is None
        or ticker is None
        or ejercicio is None
        or fuente not in {"xbrl", "ambas"}
    ):
        return {
            "aplicable": False,
            "ok": None,
            "cifra_observada": cifra,
            "ticker": ticker,
            "ejercicio": ejercicio,
        }

    hechos = xbrl[
        (xbrl["ticker"] == ticker)
        & (xbrl["fiscal_year"] == int(ejercicio))
    ]
    if hechos.empty:
        return {
            "aplicable": False,
            "ok": None,
            "cifra_observada": cifra,
            "ticker": ticker,
            "ejercicio": ejercicio,
        }

    ok = any(
        miax_s2.cuadra(float(cifra), float(real), TOLERANCIA_GUARDRAIL)
        for real in hechos["value"]
    )
    return {
        "aplicable": True,
        "ok": bool(ok),
        "cifra_observada": float(cifra),
        "ticker": ticker,
        "ejercicio": int(ejercicio),
    }


def metadata_guardrail(resultado):
    """Indica si el middleware abrió una vuelta correctiva y cómo terminó."""
    correcciones = sum(
        MARCA_GUARDRAIL in _contenido_mensaje(m)
        for m in resultado.get("messages", [])
    )
    final = validar_guardrail_numerico(resultado)
    return {
        "aplicable": final["aplicable"],
        "correction_attempted": bool(correcciones),
        "n_correcciones": int(correcciones),
        "final_ok": final["ok"],
        "validacion_final": final,
    }


# ---------- interfaz pública -------------------------------------------

def responder(pregunta: str):
    """Ejecuta una pregunta aislada contra el agente final."""
    if agente is None:
        raise RuntimeError("El agente no está disponible: configura GEMINI_API_KEY.")

    resultado = agente.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config={"configurable": {"thread_id": f"eval-{uuid.uuid4()}"}},
    )
    resultado["_guardrail_numerico"] = metadata_guardrail(resultado)
    return resultado



def responder_baseline(pregunta: str):
    """Ejecuta una pregunta aislada contra el baseline congelado."""
    if agente_baseline is None:
        raise RuntimeError(
            "El agente baseline no está disponible: configura GEMINI_API_KEY."
        )

    resultado = agente_baseline.invoke(
        {"messages": [{"role": "user", "content": pregunta}]},
        config={"configurable": {"thread_id": f"baseline-{uuid.uuid4()}"}},
    )
    resultado["_guardrail_numerico"] = metadata_guardrail(resultado)
    return resultado


def registro_resultado(resultado, latencia_s):
    """Convierte la ejecución en un registro JSON serializable."""
    uso = uso_tokens(resultado)
    return {
        "respuesta": _respuesta_dict(resultado),
        "trayectoria": trayectoria_operativa(resultado),
        "n_tool_calls_operativas": len(llamadas_operativas(resultado)),
        "n_llamadas_oficiales": len(miax_s2.herramientas_usadas(resultado)),
        **uso,
        "coste_usd": coste_oficial_usd(resultado),
        "coste_real_estimado_usd": coste_real_estimado_usd(resultado),
        "latencia_s": float(latencia_s),
        "guardrail_numerico": resultado.get("_guardrail_numerico"),
    }


## Evaluadores oficiales de S2 y diagnósticos propios

La tabla comparable con el profesor usa exactamente tres propiedades:

- `cita_correcta`
- `cifra_coincide_xbrl`
- `uso_la_tool_correcta`

`recall@5` se calcula aparte con `miax_s2.acierta()` sobre el retriever fijo,
no leyendo el chunk citado por el agente.

Debajo se conservan nuestros evaluadores diagnósticos. Pueden ser más
estrictos, pero **no sustituyen** a los oficiales.


In [21]:
import miax_s1

# ---------- corpus para citas -------------------------------------------
CHUNKS_CORPUS = [
    json.loads(linea)
    for linea in Path("corpus/chunks.jsonl").read_text(
        encoding="utf-8"
    ).splitlines()
    if linea.strip()
]
POR_ID = {c["chunk_id"]: c for c in CHUNKS_CORPUS}


def cita_correcta(item: dict, resultado: dict) -> bool | None:
    """Evaluador oficial S2: chunk real y cita respaldada por ese chunk."""
    respuesta = resultado.get("structured_response")

    if respuesta is None:
        return None if item["familia"] == "numerica" else False

    chunk_id = getattr(respuesta, "chunk_id", None)

    if not chunk_id:
        return None if item["familia"] == "numerica" else False

    if chunk_id not in POR_ID:
        return False

    cita = getattr(respuesta, "cita", None)
    if not cita:
        return False

    cita_norm = miax_s2.normalizar(str(cita))
    texto_norm = miax_s2.normalizar(POR_ID[chunk_id]["texto"])

    # S2 indica no exigir la cita completa: el modelo puede recortarla.
    muestra = cita_norm[:120]
    return bool(muestra) and muestra in texto_norm


def cifra_coincide_xbrl(item: dict, resultado: dict) -> bool | None:
    """Evaluador oficial S2: cifra contra cifra_esperada, tolerancia 1 %."""
    esperada = item.get("cifra_esperada")
    if esperada is None:
        return None

    respuesta = resultado.get("structured_response")
    if respuesta is None:
        return False

    afirmada = getattr(respuesta, "cifra", None)
    if afirmada is None:
        return False

    return miax_s2.cuadra(
        float(afirmada),
        float(esperada),
        TOLERANCIA_OFICIAL,
    )


def uso_la_tool_correcta(item: dict, resultado: dict) -> bool:
    """Evaluador oficial S2: aparecieron TODAS las tools esperadas."""
    usadas = set(miax_s2.herramientas_usadas(resultado))
    esperadas = set(item.get("herramienta_esperada") or [])
    return esperadas.issubset(usadas)


EVALUADORES_OFICIALES = {
    "cita": cita_correcta,
    "cifra": cifra_coincide_xbrl,
    "trayectoria": uso_la_tool_correcta,
}


# ---------- recall@5 oficial, independiente del agente ------------------

def recuperar_para_recall(item: dict, k: int = 5) -> list[dict]:
    """Retriever V4: dense + BM25/RRF con los filtros del caso."""
    if not item.get("ancla_texto"):
        return []

    return _buscar_hibrido_filtrado(
        item["pregunta"],
        ticker=item.get("ticker"),
        fiscal_year=item.get("fiscal_year"),
        item=item.get("item_esperado"),
        k=k,
    )


def recall_item_oficial(item: dict, k: int = 5) -> bool | None:
    if not item.get("ancla_texto"):
        return None
    return miax_s2.acierta(
        item,
        recuperar_para_recall(item, k=k),
    )


def posicion_ancla_actual(item: dict) -> int | None:
    """Puesto del ancla con el retriever híbrido activo."""
    if not item.get("ancla_texto"):
        return None

    recuperados = _buscar_hibrido_filtrado(
        item["pregunta"],
        ticker=item.get("ticker"),
        fiscal_year=item.get("fiscal_year"),
        item=item.get("item_esperado"),
        k=len(CHUNKS_CORPUS),
    )

    return miax_s2.posicion_del_ancla(
        item,
        recuperados,
    )



def recuperar_para_recall_baseline(item: dict, k: int = 5) -> list[dict]:
    """Retriever baseline: denso original con filtros de metadatos."""
    if not item.get("ancla_texto"):
        return []

    return miax_s1.buscar(
        item["pregunta"],
        ticker=item.get("ticker"),
        fiscal_year=item.get("fiscal_year"),
        item=item.get("item_esperado"),
        k=k,
    )


def recall_item_baseline(item: dict, k: int = 5) -> bool | None:
    if not item.get("ancla_texto"):
        return None
    return miax_s2.acierta(
        item,
        recuperar_para_recall_baseline(item, k=k),
    )


print("Evaluadores oficiales:", list(EVALUADORES_OFICIALES))


# ---------- diagnósticos propios --------------------------------------
import math
import re


def _valor_xbrl(ticker, fiscal_year, concept):
    filas = xbrl[
        (xbrl["ticker"] == ticker)
        & (xbrl["fiscal_year"] == int(fiscal_year))
        & (xbrl["concept"] == concept)
    ]

    if len(filas) != 1:
        raise ValueError(
            f"Se esperaba un único hecho XBRL para "
            f"{ticker} FY{fiscal_year} {concept}; encontrados: {len(filas)}"
        )

    return float(filas.iloc[0]["value"])


def evaluar_numerico_diagnostico(registro):
    """Evalúa el campo numérico estructurado frente al ground truth XBRL."""
    esperado = registro["esperado"]
    observado = registro.get("respuesta") or {}

    cifra_esperada = esperado.get("cifra")

    if cifra_esperada is None:
        return {
            "aplicable": False,
            "ok": None,
            "cifra_ok": None,
            "unidad_ok": None,
        }

    cifra_observada = observado.get("cifra")
    unidad_esperada = esperado.get("unidad")
    unidad_observada = observado.get("unidad")

    unidad_ok = (
        unidad_esperada is None
        or (
            unidad_observada is not None
            and str(unidad_observada).upper()
            == str(unidad_esperada).upper()
        )
    )

    if registro["familia"] == "numerica":
        cifra_ok = (
            cifra_observada is not None
            and math.isclose(
                float(cifra_observada),
                float(cifra_esperada),
                rel_tol=1e-9,
                abs_tol=0.01,
            )
        )

        return {
            "aplicable": True,
            "ok": cifra_ok and unidad_ok,
            "cifra_ok": cifra_ok,
            "unidad_ok": unidad_ok,
            "cifra_esperada": cifra_esperada,
            "cifra_observada": cifra_observada,
            "interpretacion": "valor",
        }

    # Comparativas: el esquema permite una sola cifra, aunque la respuesta
    # puede contener varias. Se acepta que represente el valor final o el
    # cambio absoluto entre FY anterior y FY actual.
    ticker = esperado["ticker"]
    fy_actual = int(esperado["fiscal_year"])
    fy_anterior = fy_actual - 1
    concepto = esperado["concept_xbrl"]

    valor_anterior = _valor_xbrl(ticker, fy_anterior, concepto)
    valor_actual = _valor_xbrl(ticker, fy_actual, concepto)
    cambio_absoluto = valor_actual - valor_anterior

    valor_final_ok = (
        cifra_observada is not None
        and math.isclose(
            float(cifra_observada),
            valor_actual,
            rel_tol=1e-9,
            abs_tol=0.01,
        )
    )

    cambio_ok = (
        cifra_observada is not None
        and math.isclose(
            float(cifra_observada),
            cambio_absoluto,
            rel_tol=1e-9,
            abs_tol=0.01,
        )
    )

    cifra_ok = valor_final_ok or cambio_ok

    if valor_final_ok:
        interpretacion = "valor_final"
    elif cambio_ok:
        interpretacion = "cambio_absoluto"
    else:
        interpretacion = None

    return {
        "aplicable": True,
        "ok": cifra_ok and unidad_ok,
        "cifra_ok": cifra_ok,
        "unidad_ok": unidad_ok,
        "cifra_observada": cifra_observada,
        "valor_anterior_xbrl": valor_anterior,
        "valor_actual_xbrl": valor_actual,
        "cambio_absoluto_xbrl": cambio_absoluto,
        "interpretacion": interpretacion,
    }


def chunks_recuperados(registro):
    """Extrae {chunk_id: texto} de las llamadas a search_filings."""
    chunks = {}

    for paso in registro.get("trayectoria", []):
        if paso["name"] != "search_filings":
            continue

        output = paso.get("output") or ""
        patron = r"(?ms)^\[([^\]]+)\].*?(?=^\[[^\]]+\]|\Z)"

        for match in re.finditer(patron, output):
            chunk_id = match.group(1).strip()
            chunks[chunk_id] = match.group(0)

    return chunks


def evaluar_cita_diagnostico(registro):
    """Evalúa si el retrieval recuperó el ancla y si la respuesta la citó."""
    ancla = registro["esperado"].get("ancla_texto")

    if not ancla:
        return {
            "aplicable": False,
            "retrieval_hit": None,
            "cita_ok": None,
        }

    chunks = chunks_recuperados(registro)

    chunks_con_ancla = [
        chunk_id
        for chunk_id, texto in chunks.items()
        if ancla in texto
    ]

    retrieval_hit = bool(chunks_con_ancla)
    respuesta = registro.get("respuesta") or {}

    cita_observada = respuesta.get("chunk_id")
    if not cita_observada:
        cita_observada = respuesta.get("cita")

    cita_ok = bool(
        cita_observada
        and any(
            chunk_id in str(cita_observada)
            for chunk_id in chunks_con_ancla
        )
    )

    return {
        "aplicable": True,
        "retrieval_hit": retrieval_hit,
        "cita_ok": cita_ok,
        "ancla_encontrada_en": chunks_con_ancla,
        "chunk_esperado": registro["esperado"].get("chunk_id"),
        "cita_observada": cita_observada,
    }


def _entero_seguro(valor):
    try:
        return int(valor)
    except (TypeError, ValueError):
        return None


def evaluar_trayectoria_diagnostico(registro):
    """Evalúa routing/cobertura y separa corrección de eficiencia."""
    esperado = registro["esperado"]
    familia = registro["familia"]
    trayectoria = registro.get("trayectoria", [])

    esperadas = set(esperado["herramientas"])
    auxiliares = {"list_available"}

    observadas_lista = [p["name"] for p in trayectoria]
    observadas = set(observadas_lista)

    faltantes = sorted(esperadas - observadas)
    inesperadas = sorted(observadas - esperadas - auxiliares)

    llamadas_extra = []

    # Herramientas operativas no requeridas: ineficiencia, no fallo de routing.
    for paso in trayectoria:
        if paso["name"] in inesperadas:
            llamadas_extra.append({
                "name": paso["name"],
                "args": paso.get("args", {}),
                "motivo": "herramienta operativa no necesaria para este caso",
            })

    ticker = esperado["ticker"]
    fy_actual = int(esperado["fiscal_year"])
    concepto = esperado.get("concept_xbrl")
    item = esperado.get("item")

    if familia == "comparativa":
        years_esperados = {fy_actual - 1, fy_actual}
    else:
        years_esperados = {fy_actual}

    diagnostico = {}

    # XBRL ---------------------------------------------------------------
    if "get_xbrl_fact" in esperadas:
        llamadas_xbrl = [
            p for p in trayectoria
            if p["name"] == "get_xbrl_fact"
        ]
        years_xbrl_correctos = set()

        for paso in llamadas_xbrl:
            args = paso.get("args", {})
            fy = _entero_seguro(args.get("fiscal_year"))

            llamada_correcta = (
                args.get("ticker") == ticker
                and fy in years_esperados
                and args.get("concept") == concepto
            )

            if llamada_correcta:
                years_xbrl_correctos.add(fy)
            else:
                llamadas_extra.append({
                    "name": paso["name"],
                    "args": args,
                    "motivo": "argumentos XBRL no esperados",
                })

        cobertura_xbrl_ok = (
            years_xbrl_correctos == years_esperados
        )

        diagnostico["xbrl"] = {
            "years_esperados": sorted(years_esperados),
            "years_correctos": sorted(years_xbrl_correctos),
            "cobertura_ok": cobertura_xbrl_ok,
        }
    else:
        cobertura_xbrl_ok = True

    # Retrieval textual --------------------------------------------------
    if "search_filings" in esperadas:
        llamadas_search = [
            p for p in trayectoria
            if p["name"] == "search_filings"
        ]
        years_search_correctos = set()

        for paso in llamadas_search:
            args = paso.get("args", {})
            fy = _entero_seguro(args.get("fiscal_year"))

            llamada_correcta = (
                args.get("ticker") == ticker
                and fy in years_esperados
                and (
                    item is None
                    or str(args.get("item")) == str(item)
                )
            )

            if llamada_correcta:
                years_search_correctos.add(fy)
            else:
                llamadas_extra.append({
                    "name": paso["name"],
                    "args": args,
                    "motivo": "filtros de búsqueda no esperados",
                })

        cobertura_search_ok = (
            years_search_correctos == years_esperados
        )

        diagnostico["search"] = {
            "years_esperados": sorted(years_esperados),
            "years_correctos": sorted(years_search_correctos),
            "cobertura_ok": cobertura_search_ok,
        }
    else:
        cobertura_search_ok = True

    ok = (
        not faltantes
        and cobertura_xbrl_ok
        and cobertura_search_ok
    )

    return {
        "ok": ok,
        "eficiente": len(llamadas_extra) == 0,
        "esperadas": sorted(esperadas),
        "observadas": observadas_lista,
        "faltantes": faltantes,
        "inesperadas": inesperadas,
        "diagnostico": diagnostico,
        "n_llamadas_extra": len(llamadas_extra),
        "llamadas_extra": llamadas_extra,
    }


Evaluadores oficiales: ['cita', 'cifra', 'trayectoria']


In [22]:
def _evaluar_una(caso, respondedor, recall_fn):
    """Ejecuta un caso y guarda métricas oficiales + diagnósticas."""
    inicio = time.perf_counter()

    try:
        resultado = respondedor(caso["pregunta"])
        error_ejecucion = None
    except Exception as exc:
        resultado = {
            "messages": [],
            "structured_response": None,
            "_guardrail_numerico": None,
        }
        error_ejecucion = f"{type(exc).__name__}: {exc}"

    latencia_s = time.perf_counter() - inicio

    oficial = {
        nombre: evaluador(caso, resultado)
        for nombre, evaluador in EVALUADORES_OFICIALES.items()
    }
    oficial["recall@5"] = recall_fn(caso, k=5)

    registro = {
        "id": caso["id"],
        "familia": caso["familia"],
        "pregunta": caso["pregunta"],
        "esperado": {
            "ticker": caso["ticker"],
            "fiscal_year": caso["fiscal_year"],
            "respuesta": caso["respuesta_esperada"],
            "cifra": caso["cifra_esperada"],
            "unidad": caso["unidad"],
            "concept_xbrl": caso["concept_xbrl"],
            "item": caso["item_esperado"],
            "ancla_texto": caso["ancla_texto"],
            "ancla_inicio": caso["ancla_inicio"],
            "ancla_fin": caso["ancla_fin"],
            "chunk_id": caso["chunk_id_esperado"],
            "herramientas": caso["herramienta_esperada"],
        },
        **registro_resultado(resultado, latencia_s),
        "error_ejecucion": error_ejecucion,
        "evaluacion_oficial": oficial,
    }

    registro["evaluacion_diagnostica"] = {
        "numerico": evaluar_numerico_diagnostico(registro),
        "cita_retrieval": evaluar_cita_diagnostico(registro),
        "trayectoria": evaluar_trayectoria_diagnostico(registro),
    }
    return registro


def _acierto_oficial(registro):
    if registro.get("error_ejecucion"):
        return False

    valores = [
        registro["evaluacion_oficial"]["cita"],
        registro["evaluacion_oficial"]["cifra"],
        registro["evaluacion_oficial"]["trayectoria"],
    ]
    aplicables = [v for v in valores if v is not None]
    return bool(aplicables) and all(bool(v) for v in aplicables)


def _evaluar_ruta(ruta_jsonl, respondedor, recall_fn):
    ruta = Path(ruta_jsonl)
    casos = leer_jsonl(ruta)

    if not casos:
        raise ValueError(f"No hay casos en {ruta}")

    problemas = validar(casos, exigir_20=False)
    if problemas:
        raise ValueError(
            "El JSONL no pasa el validador:\n- "
            + "\n- ".join(problemas)
        )

    registros = [
        _evaluar_una(caso, respondedor, recall_fn)
        for caso in casos
    ]
    filas = []

    for r in registros:
        of = r["evaluacion_oficial"]
        diag = r["evaluacion_diagnostica"]
        guardrail = r.get("guardrail_numerico") or {}

        filas.append({
            "id": r["id"],
            "familia": r["familia"],
            "ticker": r["esperado"]["ticker"],
            "acierto_oficial": _acierto_oficial(r),
            "cita": of["cita"],
            "cifra": of["cifra"],
            "trayectoria": of["trayectoria"],
            "recall": of["recall@5"],
            "coste_usd": r["coste_usd"],
            "latencia_s": r["latencia_s"],
            "llamadas": r["n_llamadas_oficiales"],
            "retrieval_hit_trayectoria": (
                diag["cita_retrieval"].get("retrieval_hit")
            ),
            "cita_trayectoria_ok": (
                diag["cita_retrieval"].get("cita_ok")
            ),
            "numerico_diagnostico_ok": diag["numerico"].get("ok"),
            "trayectoria_diagnostica_ok": diag["trayectoria"].get("ok"),
            "trayectoria_eficiente": diag["trayectoria"].get("eficiente"),
            "n_tool_calls_operativas": r["n_tool_calls_operativas"],
            "model_calls": r["model_calls"],
            "input_tokens": r["input_tokens"],
            "cache_read_tokens": r["cache_read_tokens"],
            "output_tokens": r["output_tokens"],
            "total_tokens": r["total_tokens"],
            "coste_real_estimado_usd": r["coste_real_estimado_usd"],
            "guardrail_aplicable": guardrail.get("aplicable"),
            "guardrail_correction_attempted": guardrail.get(
                "correction_attempted"
            ),
            "guardrail_n_correcciones": guardrail.get("n_correcciones"),
            "guardrail_final_ok": guardrail.get("final_ok"),
            "error_ejecucion": r["error_ejecucion"],
        })

    return registros, pd.DataFrame(filas)


def evaluar(ruta_jsonl):
    """Interfaz pública final para golden y holdout."""
    return _evaluar_ruta(
        ruta_jsonl,
        respondedor=responder,
        recall_fn=recall_item_oficial,
    )


def evaluar_baseline(ruta_jsonl):
    """Regenera el baseline denso con las mismas métricas oficiales."""
    return _evaluar_ruta(
        ruta_jsonl,
        respondedor=responder_baseline,
        recall_fn=recall_item_baseline,
    )


def resumir_oficial(tabla: pd.DataFrame, etiqueta: str) -> dict:
    def tasa(columna):
        valores = tabla[columna].dropna() if columna in tabla else []
        return float(valores.mean()) if len(valores) else float("nan")

    return {
        "versión": etiqueta,
        "cita": tasa("cita"),
        "cifra": tasa("cifra"),
        "trayectoria": tasa("trayectoria"),
        "recall@5": tasa("recall"),
        "coste medio (¢)": tabla["coste_usd"].mean() * 100,
        "latencia media (s)": tabla["latencia_s"].mean(),
        "llamadas/pregunta": tabla["llamadas"].mean(),
    }


def resumir_por_familia(tabla: pd.DataFrame) -> pd.DataFrame:
    filas = []

    grupos = [("TOTAL", tabla)] + [
        (f, g) for f, g in tabla.groupby("familia", sort=True)
    ]

    for nombre, grupo in grupos:
        filas.append({
            "familia": nombre,
            "n": len(grupo),
            "aciertos": int(grupo["acierto_oficial"].sum()),
            "accuracy": float(grupo["acierto_oficial"].mean()),
            "cita": (
                float(grupo["cita"].dropna().mean())
                if len(grupo["cita"].dropna()) else None
            ),
            "cifra": (
                float(grupo["cifra"].dropna().mean())
                if len(grupo["cifra"].dropna()) else None
            ),
            "trayectoria": float(grupo["trayectoria"].mean()),
            "recall@5": (
                float(grupo["recall"].dropna().mean())
                if len(grupo["recall"].dropna()) else None
            ),
            "coste_medio_usd": float(grupo["coste_usd"].mean()),
            "latencia_media_s": float(grupo["latencia_s"].mean()),
            "llamadas_media": float(grupo["llamadas"].mean()),
            "tools_operativas_media": float(
                grupo["n_tool_calls_operativas"].mean()
            ),
        })

    return pd.DataFrame(filas)


def ejecutar_y_guardar(
    ruta_jsonl,
    etiqueta,
    prefijo,
    funcion_evaluar=evaluar,
    forzar=False,
):
    """Carga resultados existentes o los regenera si faltan."""
    ruta_resultados = Path("resultados")
    ruta_resultados.mkdir(parents=True, exist_ok=True)

    ruta_registros = ruta_resultados / f"{prefijo}_registros.jsonl"
    ruta_tabla = ruta_resultados / f"{prefijo}_tabla.csv"
    ruta_metricas = ruta_resultados / f"{prefijo}_metricas.csv"

    if (
        ruta_registros.is_file()
        and ruta_tabla.is_file()
        and not forzar
    ):
        registros = leer_jsonl(ruta_registros)
        tabla = pd.read_csv(ruta_tabla)
        metricas = (
            pd.read_csv(ruta_metricas)
            if ruta_metricas.is_file()
            else resumir_por_familia(tabla)
        )
        print(
            f"{etiqueta}: resultados existentes cargados; "
            "no se llamó a Gemini."
        )
        return registros, tabla, metricas

    print(f"{etiqueta}: no hay resultados completos; regenerando...")
    registros, tabla = funcion_evaluar(ruta_jsonl)
    metricas = resumir_por_familia(tabla)

    with ruta_registros.open("w", encoding="utf-8") as f:
        for registro in registros:
            f.write(
                json.dumps(
                    registro,
                    ensure_ascii=False,
                    allow_nan=False,
                )
                + "\n"
            )

    tabla.to_csv(ruta_tabla, index=False)
    metricas.to_csv(ruta_metricas, index=False)

    print(f"{etiqueta}: ejecutado y guardado en {ruta_resultados}.")
    return registros, tabla, metricas


print("Evaluación baseline/final preparada.")


Evaluación baseline/final preparada.


## Resultados reproducibles

En un clon limpio, `Run all` crea `resultados/` y regenera automáticamente
**baseline** y **sistema final** si sus ficheros no existen.

Si los resultados ya están en el repositorio, se cargan y no se vuelve a gastar
API. Así los ficheros exigidos por la práctica son persistentes y reproducibles.


In [23]:
# Baseline y final del golden propio: se regeneran solo si faltan.
RUTA_RESULTADOS = Path("resultados")
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

registros_baseline, tabla_baseline, metricas_baseline = ejecutar_y_guardar(
    RUTA_GOLDEN_PROPIO,
    etiqueta="Baseline · golden propio",
    prefijo="baseline",
    funcion_evaluar=evaluar_baseline,
    forzar=False,
)

registros_final, tabla_final, metricas_final = ejecutar_y_guardar(
    RUTA_GOLDEN_PROPIO,
    etiqueta="Final V4 · golden propio",
    prefijo="v4_hibrido_propio",
    funcion_evaluar=evaluar,
    forzar=False,
)

print("\nContenido de resultados/:")
for p in sorted(RUTA_RESULTADOS.iterdir()):
    print(" -", p.name)


Baseline · golden propio: no hay resultados completos; regenerando...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Baseline · golden propio: ejecutado y guardado en resultados.
Final V4 · golden propio: no hay resultados completos; regenerando...
Final V4 · golden propio: ejecutado y guardado en resultados.

Contenido de resultados/:
 - baseline_metricas.csv
 - baseline_registros.jsonl
 - baseline_tabla.csv
 - v4_hibrido_propio_metricas.csv
 - v4_hibrido_propio_registros.jsonl
 - v4_hibrido_propio_tabla.csv


In [24]:
# Benchmark oficial del profesor: opcional, no necesario para el entregable.
EJECUTAR_GOLDEN_OFICIAL = False

if golden_oficial and EJECUTAR_GOLDEN_OFICIAL:
    ejecutar_y_guardar(
        RUTA_GOLDEN_OFICIAL,
        etiqueta="Final V4 · golden oficial",
        prefijo="v4_hibrido_oficial",
        funcion_evaluar=evaluar,
        forzar=False,
    )
elif golden_oficial:
    print("Golden oficial disponible; benchmark opcional no relanzado.")
else:
    print("Golden oficial no presente: correcto, no es dependencia de la entrega.")


Golden oficial disponible; benchmark opcional no relanzado.


## Comparación formal baseline vs final

La tabla exigida se regenera desde los registros guardados del **golden propio**. El loader acepta además el antiguo `baseline_registros.jsonl` que quedó con separadores `\n` literales y lo normaliza a JSONL estándar.

Para coste se aplica a ambas versiones la misma fórmula S2 (input + output, sin descuento de caché). `recall@5` se calcula aisladamente sobre la pregunta original: dense baseline frente a híbrido final.


In [25]:
def _acierto_guardado(r: dict) -> bool:
    if r.get("error_ejecucion"):
        return False
    of = r["evaluacion_oficial"]
    aplicables = [
        of[k] for k in ("cita", "cifra", "trayectoria")
        if of[k] is not None
    ]
    return bool(aplicables) and all(bool(v) for v in aplicables)


def _resumen_registros(registros: list[dict], version: str) -> pd.DataFrame:
    filas = []

    for r in registros:
        filas.append({
            "version": version,
            "familia": r["familia"],
            "acierto": _acierto_guardado(r),
            "recall": r["evaluacion_oficial"].get("recall@5"),
            "coste": float(r["coste_usd"]),
            "latencia": float(r["latencia_s"]),
            "llamadas": float(r["n_llamadas_oficiales"]),
        })

    return pd.DataFrame(filas)


df_base = _resumen_registros(registros_baseline, "Baseline")
df_final = _resumen_registros(registros_final, "Final V4")

comparacion_familias = pd.DataFrame({
    "Baseline": df_base.groupby("familia")["acierto"].agg(
        lambda x: f"{int(x.sum())}/{len(x)}"
    ),
    "Final V4": df_final.groupby("familia")["acierto"].agg(
        lambda x: f"{int(x.sum())}/{len(x)}"
    ),
}).T

comparacion_global = pd.DataFrame([
    {
        "versión": "Baseline",
        "accuracy": df_base["acierto"].mean(),
        "recall@5": df_base["recall"].dropna().mean(),
        "coste_medio_usd": df_base["coste"].mean(),
        "latencia_media_s": df_base["latencia"].mean(),
        "llamadas_por_pregunta": df_base["llamadas"].mean(),
    },
    {
        "versión": "Final V4",
        "accuracy": df_final["acierto"].mean(),
        "recall@5": df_final["recall"].dropna().mean(),
        "coste_medio_usd": df_final["coste"].mean(),
        "latencia_media_s": df_final["latencia"].mean(),
        "llamadas_por_pregunta": df_final["llamadas"].mean(),
    },
])

print("Aciertos por familia")
display(comparacion_familias)

print("Comparación global baseline vs final")
display(
    comparacion_global.style
    .format({
        "accuracy": "{:.1%}",
        "recall@5": "{:.1%}",
        "coste_medio_usd": "${:.4f}",
        "latencia_media_s": "{:.2f}",
        "llamadas_por_pregunta": "{:.2f}",
    })
    .highlight_max(subset=["accuracy", "recall@5"], axis=0)
    .highlight_min(
        subset=[
            "coste_medio_usd",
            "latencia_media_s",
            "llamadas_por_pregunta",
        ],
        axis=0,
    )
)

comparacion_familias.to_csv(
    RUTA_RESULTADOS / "comparacion_aciertos_familia.csv"
)
comparacion_global.to_csv(
    RUTA_RESULTADOS / "comparacion_baseline_final.csv",
    index=False,
)

print("Tablas de comparación guardadas en resultados/.")


Aciertos por familia


familia,comparativa,extractiva,numerica
Baseline,6/6,7/7,4/7
Final V4,5/6,7/7,7/7


Comparación global baseline vs final


,versión,accuracy,recall@5,coste_medio_usd,latencia_media_s,llamadas_por_pregunta
0,Baseline,85.0%,69.2%,$0.0187,10.64,4.95
1,Final V4,95.0%,61.5%,$0.0173,8.26,4.30


Tablas de comparación guardadas en resultados/.


## Estado final antes del holdout

`Run all` deja ahora:

- agente baseline reproducible;
- agente final V4;
- `resultados/baseline_*`;
- `resultados/v4_hibrido_propio_*`;
- tabla automática baseline vs final;
- cuatro tools contractuales;
- salida estructurada;
- guardrail numérico `after_model`;
- tres evaluadores;
- `responder(pregunta)` y `evaluar(ruta_jsonl)`.

El holdout de 10 preguntas se ejecutará en la exposición sin modificar código:

```python
registros_holdout, tabla_holdout = evaluar("holdout.jsonl")
```
